# Exploratory Data Analysis - Crypto Price Prediction

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Ensure the project root is importable
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.utils.config import load_config
from src.utils.io import load_dataframe
from src.utils.constants import *

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

## 1. Load Data

In [ ]:
config = load_config()

# Adjust the path to point at the processed parquet you want to explore.
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "BTC_USDT_1h.parquet"

df = load_dataframe(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min()} -> {df.index.max()}")
df.head(10)

## 2. Basic Statistics

In [ ]:
df.describe()

In [ ]:
# Check for missing values
null_counts = df.isnull().sum()
null_pct = (df.isnull().mean() * 100).round(2)
null_info = pd.DataFrame({"null_count": null_counts, "null_pct": null_pct})
null_info[null_info["null_count"] > 0]

## 3. Price Distribution

In [ ]:
# Close-price histogram
fig = px.histogram(
    df,
    x="close",
    nbins=100,
    title="Close Price Distribution",
    labels={"close": "Close Price (USDT)"},
    marginal="box",
)
fig.show()

In [ ]:
# Close-price time series
fig = px.line(
    df,
    y="close",
    title="Close Price Over Time",
    labels={"close": "Close Price (USDT)", "index": "Date"},
)
fig.update_layout(xaxis_title="Date", yaxis_title="Price (USDT)")
fig.show()

## 4. Technical Indicators

In [ ]:
fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=("RSI (14)", "MACD", "Bollinger Bands"),
    row_heights=[0.3, 0.3, 0.4],
)

# --- RSI ---
if "rsi" in df.columns:
    fig.add_trace(
        go.Scatter(x=df.index, y=df["rsi"], name="RSI", line=dict(color="purple")),
        row=1, col=1,
    )
    fig.add_hline(y=70, line_dash="dash", line_color="red", row=1, col=1)
    fig.add_hline(y=30, line_dash="dash", line_color="green", row=1, col=1)

# --- MACD ---
if "macd" in df.columns:
    fig.add_trace(
        go.Scatter(x=df.index, y=df["macd"], name="MACD", line=dict(color="blue")),
        row=2, col=1,
    )
    if "macd_signal" in df.columns:
        fig.add_trace(
            go.Scatter(x=df.index, y=df["macd_signal"], name="Signal", line=dict(color="orange")),
            row=2, col=1,
        )
    if "macd_hist" in df.columns:
        colors = ["green" if v >= 0 else "red" for v in df["macd_hist"]]
        fig.add_trace(
            go.Bar(x=df.index, y=df["macd_hist"], name="Histogram", marker_color=colors),
            row=2, col=1,
        )

# --- Bollinger Bands ---
fig.add_trace(
    go.Scatter(x=df.index, y=df["close"], name="Close", line=dict(color="black")),
    row=3, col=1,
)
if "bb_upper" in df.columns:
    fig.add_trace(
        go.Scatter(x=df.index, y=df["bb_upper"], name="BB Upper",
                   line=dict(color="grey", dash="dash")),
        row=3, col=1,
    )
if "bb_lower" in df.columns:
    fig.add_trace(
        go.Scatter(x=df.index, y=df["bb_lower"], name="BB Lower",
                   line=dict(color="grey", dash="dash"),
                   fill="tonexty", fillcolor="rgba(128,128,128,0.1)"),
        row=3, col=1,
    )

fig.update_layout(height=900, title_text="Technical Indicators", showlegend=True)
fig.show()

## 5. Correlation Analysis

In [ ]:
# Select only numeric columns for the correlation matrix
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Feature Correlation Heatmap",
    aspect="auto",
)
fig.update_layout(width=1000, height=900)
fig.show()

## 6. Stationarity Tests

In [ ]:
from statsmodels.tsa.stattools import adfuller


def run_adf_test(series: pd.Series, name: str) -> dict:
    """Run Augmented Dickey-Fuller test and return a summary dict."""
    result = adfuller(series.dropna(), autolag="AIC")
    return {
        "Series": name,
        "ADF Statistic": result[0],
        "p-value": result[1],
        "Lags Used": result[2],
        "Observations": result[3],
        "Stationary (p < 0.05)": result[1] < 0.05,
    }


# ADF on raw close prices
adf_close = run_adf_test(df["close"], "Close Price")

# ADF on log returns
log_returns = np.log(df["close"] / df["close"].shift(1))
adf_logret = run_adf_test(log_returns, "Log Returns")

adf_df = pd.DataFrame([adf_close, adf_logret])
adf_df

## 7. Target Distribution

In [ ]:
# Compute simple 1-step returns and direction labels
df["return_1"] = df["close"].pct_change()
df["direction"] = (df["return_1"] > 0).astype(int).map({1: "Up", 0: "Down"})

# Direction label counts
direction_counts = df["direction"].value_counts()
print("Direction label counts:")
print(direction_counts)
print(f"\nUp ratio: {(direction_counts.get('Up', 0) / len(df.dropna()) * 100):.1f}%")

In [ ]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Direction Label Distribution", "Return Distribution"),
)

# Bar chart of direction labels
fig.add_trace(
    go.Bar(
        x=direction_counts.index.tolist(),
        y=direction_counts.values.tolist(),
        marker_color=["green", "red"],
        name="Direction",
    ),
    row=1, col=1,
)

# Histogram of returns
returns_clean = df["return_1"].dropna()
fig.add_trace(
    go.Histogram(
        x=returns_clean,
        nbinsx=150,
        name="Returns",
        marker_color="steelblue",
    ),
    row=1, col=2,
)

fig.update_layout(
    height=450,
    title_text="Target Analysis",
    showlegend=False,
)
fig.update_xaxes(title_text="Direction", row=1, col=1)
fig.update_xaxes(title_text="1-step Return", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.show()